# 03 — Traditional-feature similarity baseline

**Goal:** Retrieve similar songs using standardized FMA features and evaluate genre consistency. All retrieval logic comes from `scripts/baseline.py` rather than being duplicated here. Test tracks are queries; training tracks are candidates.

## 1. Colab / local repository setup
Set `REPO_URL` to your GitHub repository URL when using Colab. If the repository is already cloned, this cell uses it. For the latest changes, restart the runtime or run `git pull` manually.

In [ ]:
from pathlib import Path
import os, sys, subprocess

REPO_NAME = "music-similarity-representation-learning"
REPO_URL = "https://github.com/samiyaalizaidi/music-similarity-representation-learning.git"

if Path('/content').exists():  # Colab
    repo = Path('/content') / REPO_NAME
    if not repo.exists():
        if 'YOUR_USERNAME' in REPO_URL:
            raise ValueError('Replace YOUR_USERNAME in REPO_URL with your GitHub username.')
        subprocess.run(['git', 'clone', REPO_URL, str(repo)], check=True)
    PROJECT_ROOT = repo
else:
    cwd = Path.cwd().resolve()
    PROJECT_ROOT = next((p for p in [cwd, *cwd.parents] if (p / 'scripts' / 'baseline.py').is_file()), None)
    if PROJECT_ROOT is None:
        raise FileNotFoundError('Run the notebook inside your repository.')
os.chdir(PROJECT_ROOT)
print('Project root:', PROJECT_ROOT)
print('Baseline script:', (PROJECT_ROOT / 'scripts' / 'baseline.py').exists())

## 2. Install and import dependencies

In [ ]:
%pip -q install numpy pandas scikit-learn matplotlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
sys.path.insert(0, str(PROJECT_ROOT / 'scripts'))
from baseline import load_data, build_index, retrieve, evaluate_genre_consistency

## 3. Load and validate processed data
No preprocessing is rerun. The files must already be committed under `data/processed/`.

In [ ]:
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
required = ['all.npz', 'metadata.csv']
missing = [name for name in required if not (PROCESSED_DIR / name).exists()]
if missing:
    raise FileNotFoundError(f'Missing processed files: {missing}. Pull the latest GitHub commit.')
X, track_ids, splits, metadata = load_data(PROCESSED_DIR)
print('Feature matrix:', X.shape)
print('Split counts:', pd.Series(splits).value_counts().to_dict())
print('Metadata columns:', list(metadata.columns))
print('Finite features:', bool(np.isfinite(X).all()))
display(metadata.head())

## 4. Build the baseline nearest-neighbor index
The candidate pool is restricted to training tracks. The evaluation queries come from the test split.

In [ ]:
K = 5
model, candidate_indices = build_index(X, splits, k=K)
test_indices = np.flatnonzero(splits == 'test')
print('Training candidates:', len(candidate_indices))
print('Test queries:', len(test_indices))

## 5. Inspect three example queries

In [ ]:
example_indices = test_indices[:3]
for idx in example_indices:
    qid = int(track_ids[idx])
    print('\nQUERY:', qid)
    display(metadata.loc[[qid], ['title', 'artist', 'genre']])
    results = retrieve(X, track_ids, splits, metadata, model, candidate_indices, int(idx), k=K)
    display(results[['rank', 'track_id', 'title', 'artist', 'genre', 'cosine_similarity', 'same_genre']].round(3))

## 6. Evaluate genre consistency
Genre consistency is a **proxy**, not a human-judged measure of musical similarity. A cross-genre match is not automatically wrong.

In [ ]:
metrics = [evaluate_genre_consistency(X, track_ids, splits, metadata, model, candidate_indices, k=k) for k in (5, 10)]
metrics_df = pd.DataFrame(metrics)
display(metrics_df.round(4))

## 7. Plot preliminary metrics

In [ ]:
ax = metrics_df.plot(x='k', y=['genre_consistency', 'cross_genre_rate'], kind='bar', figsize=(7, 4), rot=0)
ax.set_xlabel('Number of neighbors (k)')
ax.set_ylabel('Mean proportion')
ax.set_ylim(0, 1)
ax.set_title('Traditional-feature baseline retrieval')
plt.tight_layout()
plt.show()

## 8. Save results for the milestone report

In [ ]:
RESULTS_DIR = PROJECT_ROOT / 'results'
RESULTS_DIR.mkdir(exist_ok=True)
metrics_df.to_csv(RESULTS_DIR / 'baseline_metrics.csv', index=False)
example_results = pd.concat([retrieve(X, track_ids, splits, metadata, model, candidate_indices, int(idx), k=K) for idx in example_indices], ignore_index=True)
example_results.to_csv(RESULTS_DIR / 'baseline_examples.csv', index=False)
print('Saved:', RESULTS_DIR / 'baseline_metrics.csv')
print('Saved:', RESULTS_DIR / 'baseline_examples.csv')

## 9. Comparison with the autoencoder
Use the **same test queries, training candidate pool, cosine distance, and k values** with learned embeddings. Compare genre consistency and inspect cross-genre examples; do not treat genre agreement as ground-truth similarity.